<a href="https://colab.research.google.com/github/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/blob/main/SI26_Week1_Humna_Imran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import os
import shutil
import subprocess
import zipfile

In [2]:
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"

In [3]:
DOWNLOAD_DIR = "utrset_real_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")

In [4]:
OUT_DIR = "data/raw/other"
LABELS_CSV = "data/labels.csv"
N_SAMPLES = 60

In [5]:
def download():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return

    print("Downloading UTRSet-Real from Google Drive (this is ~hundreds of MB, may take a few minutes)...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    except ImportError:
        # fallback to gdown CLI if the python import path is unavailable
        subprocess.run(
            ["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH],
            check=True,
        )

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed or produced an empty file. Google Drive "
            "sometimes blocks automated downloads of large files with a "
            "'too many downloads' warning page instead of the real file. "
            "If this happens: open the link in your own browser "
            "(https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view), "
            "click through the warning, download manually, then upload "
            "the zip to Colab and re-run this script -- it'll detect the "
            "existing zip and skip straight to extraction."
        )
    print(f"Downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")

In [6]:
def extract():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")

In [7]:
def find_gt_file(root):
    """Locate the ground-truth label file -- commonly gt.txt per the
    dataset author's own repo convention, but search broadly in case
    the zip structure differs."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates

In [8]:
def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images

In [9]:
def main():
    download()
    extract()

    gt_files = find_gt_file(EXTRACT_DIR)
    images = find_images(EXTRACT_DIR)

    print(f"\nFound {len(gt_files)} ground-truth file(s), {len(images)} image(s) in the extracted archive.")

    os.makedirs(OUT_DIR, exist_ok=True)

    new_rows = []

    if gt_files:
        gt_path = gt_files[0]
        print(f"Parsing ground truth from: {gt_path}")
        with open(gt_path, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]

        gt_dir = os.path.dirname(gt_path)
        count = 0
        for line in lines:
            if count >= N_SAMPLES:
                break
            if "\t" in line:
                img_rel_path, text = line.split("\t", 1)
            elif " " in line:
                img_rel_path, text = line.split(" ", 1)
            else:
                continue

            src_img_path = os.path.join(gt_dir, img_rel_path)
            if not os.path.exists(src_img_path):
                src_img_path = os.path.join(EXTRACT_DIR, img_rel_path)
            if not os.path.exists(src_img_path):
                continue

            fname = f"utrset_{count:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(src_img_path, dst_path)
            new_rows.append({"image": dst_path, "text": text})
            count += 1

        print(f"Copied {count} images with verified ground-truth labels.")

    elif images:
        print("WARNING: No ground-truth file found automatically.")
        print("Copying a sample of images anyway -- you'll need to find")
        print("the label source manually (check the extracted folder")
        print(f"structure at {EXTRACT_DIR}) or label these by eye.")
        for i, img_path in enumerate(images[:N_SAMPLES]):
            fname = f"utrset_{i:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(img_path, dst_path)
            new_rows.append({"image": dst_path, "text": "FILL_IN_URDU_TEXT_HERE"})
    else:
        raise RuntimeError(
            f"No images or ground-truth files found under {EXTRACT_DIR}. "
            f"Check the actual folder structure with: "
            f"!find {EXTRACT_DIR} -maxdepth 3"
        )

    existing_rows = []
    if os.path.exists(LABELS_CSV):
        with open(LABELS_CSV, "r", encoding="utf-8") as f:
            existing_rows = list(csv.DictReader(f))
        print(f"\nFound existing labels.csv with {len(existing_rows)} rows -- appending.")
    else:
        os.makedirs(os.path.dirname(LABELS_CSV), exist_ok=True)

    all_rows = existing_rows + new_rows
    with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"labels.csv now has {len(all_rows)} total entries.")
    print(f"\nDataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).")
    print("Cite in your README:")
    print("""
  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
""")


if __name__ == "__main__":
    main()

Downloading...
From (original): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T
From (redirected): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T&confirm=t&uuid=8c15541d-c651-405a-b364-88c277f6008c
To: /content/utrset_real_download/utrset_real.zip
100%|██████████| 212M/212M [00:02<00:00, 99.1MB/s]


Downloaded: utrset_real_download/utrset_real.zip (211.7 MB)
Extracting...
Extracted.

Found 2 ground-truth file(s), 22322 image(s) in the extracted archive.
Parsing ground truth from: utrset_real_download/extracted/UTRSet-Real/test/gt.txt
Copied 60 images with verified ground-truth labels.
labels.csv now has 60 total entries.

Dataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).
Cite in your README:

  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.



In [10]:
!pip install Pillow arabic-reshaper python-bidi gdown -q
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 14.0 MB/s eta 0:00:00
Dependencies installed.


In [11]:
import os

folders = [
    'data/raw/newspaper',
    'data/raw/books',
    'data/raw/signboards',
    'data/raw/synthetic',
    'data/raw/other',
]

In [12]:
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('Folder structure ready!')

Created: data/raw/newspaper
Created: data/raw/books
Created: data/raw/signboards
Created: data/raw/synthetic
Created: data/raw/other
Folder structure ready!


In [13]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import os
import urllib.request
import csv

In [14]:
urdu_texts = [
    "پاکستان زندہ باد",
    "آج کا موسم خوشگوار ہے",
    "تعلیم ہر انسان کا حق ہے",
    "کراچی پاکستان کا سب سے بڑا شہر ہے",
    "محنت کامیابی کی کنجی ہے",
    "علم انسان کو روشنی دیتا ہے",
    "وقت کی قدر کرنی چاہیے",
    "صحت سب سے بڑی نعمت ہے",
    "دوستی ایک قیمتی رشتہ ہے",
    "محنت کبھی رائیگاں نہیں جاتی",
]

In [15]:
os.makedirs('data/raw/synthetic', exist_ok=True)

In [16]:
FONT_PATH = "NotoNastaliqUrdu-Regular.ttf"
if not os.path.exists(FONT_PATH):
    font_url = (
        "https://raw.githubusercontent.com/google/fonts/main/ofl/"
        "notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    try:
        urllib.request.urlretrieve(font_url, FONT_PATH)
        print(f"Downloaded font to {FONT_PATH}")
    except Exception as e:
        print(f"Could not auto-download font ({e}).")
        print("Manually download 'Noto Nastaliq Urdu' from fonts.google.com,")
        print(f"upload it to Colab's file panel, named {FONT_PATH}")

Downloaded font to NotoNastaliqUrdu-Regular.ttf


In [17]:
font = ImageFont.truetype(FONT_PATH, 36)

In [18]:
synthetic_labels = []
for i, text in enumerate(urdu_texts):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    dummy_img = Image.new('RGB', (10, 10))
    dummy_draw = ImageDraw.Draw(dummy_img)
    bbox = dummy_draw.textbbox((0, 0), bidi_text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    img = Image.new('RGB', (text_w + 40, text_h + 40), color='white')
    draw = ImageDraw.Draw(img)
    draw.text((20 - bbox[0], 20 - bbox[1]), bidi_text, fill='black', font=font)

    save_path = f'data/raw/synthetic/urdu_{i+1}.png'
    img.save(save_path)
    synthetic_labels.append({'image': save_path, 'text': text})
    print(f'Generated: {save_path} -- {text}')

Generated: data/raw/synthetic/urdu_1.png -- پاکستان زندہ باد
Generated: data/raw/synthetic/urdu_2.png -- آج کا موسم خوشگوار ہے
Generated: data/raw/synthetic/urdu_3.png -- تعلیم ہر انسان کا حق ہے
Generated: data/raw/synthetic/urdu_4.png -- کراچی پاکستان کا سب سے بڑا شہر ہے
Generated: data/raw/synthetic/urdu_5.png -- محنت کامیابی کی کنجی ہے
Generated: data/raw/synthetic/urdu_6.png -- علم انسان کو روشنی دیتا ہے
Generated: data/raw/synthetic/urdu_7.png -- وقت کی قدر کرنی چاہیے
Generated: data/raw/synthetic/urdu_8.png -- صحت سب سے بڑی نعمت ہے
Generated: data/raw/synthetic/urdu_9.png -- دوستی ایک قیمتی رشتہ ہے
Generated: data/raw/synthetic/urdu_10.png -- محنت کبھی رائیگاں نہیں جاتی


In [19]:
print(f'Done! {len(synthetic_labels)} synthetic images in data/raw/synthetic/')

Done! 10 synthetic images in data/raw/synthetic/


In [20]:
LABELS_CSV = 'data/labels.csv'
existing_rows = []
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, 'r', encoding='utf-8') as f:
        existing_rows = list(csv.DictReader(f))

all_rows = existing_rows + synthetic_labels
os.makedirs('data', exist_ok=True)
with open(LABELS_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['image', 'text'])
    writer.writeheader()
    writer.writerows(all_rows)

print(f'labels.csv now has {len(all_rows)} total entries.')

labels.csv now has 70 total entries.


In [21]:
import csv
import os
import shutil
import subprocess
import zipfile

In [22]:
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"
DOWNLOAD_DIR = "utrset_real_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")
OUT_DIR = "data/raw/other"
LABELS_CSV = "data/labels.csv"
N_SAMPLES = 60

In [23]:
def download():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return
    print("Downloading UTRSet-Real from Google Drive...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    except ImportError:
        subprocess.run(["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH], check=True)
    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed. Try opening "
            "https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view "
            "in your browser, download manually, then upload the zip to "
            f"Colab as {ZIP_PATH} and re-run this cell."
        )
    print(f"Downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")

In [24]:
def extract():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")

In [25]:
def find_gt_file(root):
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates

In [26]:
def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images

In [27]:
download()
extract()

Already downloaded: utrset_real_download/utrset_real.zip
Already extracted: utrset_real_download/extracted


In [28]:
gt_files = find_gt_file(EXTRACT_DIR)
images = find_images(EXTRACT_DIR)
print(f"\nFound {len(gt_files)} ground-truth file(s), {len(images)} image(s).")


Found 2 ground-truth file(s), 22322 image(s).


In [29]:
os.makedirs(OUT_DIR, exist_ok=True)
new_rows = []

In [30]:
if gt_files:
    gt_path = gt_files[0]
    print(f"Parsing ground truth from: {gt_path}")
    with open(gt_path, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    gt_dir = os.path.dirname(gt_path)
    count = 0
    for line in lines:
        if count >= N_SAMPLES:
            break
        if "\t" in line:
            img_rel_path, text = line.split("\t", 1)
        elif " " in line:
            img_rel_path, text = line.split(" ", 1)
        else:
            continue

        src_img_path = os.path.join(gt_dir, img_rel_path)
        if not os.path.exists(src_img_path):
            src_img_path = os.path.join(EXTRACT_DIR, img_rel_path)
        if not os.path.exists(src_img_path):
            continue

        fname = f"utrset_{count:03d}.png"
        dst_path = os.path.join(OUT_DIR, fname)
        shutil.copy(src_img_path, dst_path)
        new_rows.append({"image": dst_path, "text": text})
        count += 1

    print(f"Copied {count} images with verified ground-truth labels.")
elif images:
    print("WARNING: No ground-truth file found. Copying images with placeholders.")
    for i, img_path in enumerate(images[:N_SAMPLES]):
        fname = f"utrset_{i:03d}.png"
        dst_path = os.path.join(OUT_DIR, fname)
        shutil.copy(img_path, dst_path)
        new_rows.append({"image": dst_path, "text": "FILL_IN_URDU_TEXT_HERE"})
else:
    raise RuntimeError(f"No images/labels found under {EXTRACT_DIR}.")

Parsing ground truth from: utrset_real_download/extracted/UTRSet-Real/test/gt.txt
Copied 60 images with verified ground-truth labels.


In [31]:
existing_rows = []
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        existing_rows = list(csv.DictReader(f))

In [32]:
all_rows = existing_rows + new_rows
with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "text"])
    writer.writeheader()
    writer.writerows(all_rows)

In [33]:
print(f"labels.csv now has {len(all_rows)} total entries.")
print("\nCitation for your README:")
print("""
  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
  Dataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).
""")

labels.csv now has 130 total entries.

Citation for your README:

  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
  Dataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).



In [34]:
import os
import shutil
import zipfile

# List of your categories
category_to_folder = {"Book": "books", "Newspaper": "newspaper", "Signboard": "signboards"}
for category, folder_name in category_to_folder.items():
    zip_path = f"{category}.zip"  # Make sure these are uploaded to Colab
    extract_to = f"{category.lower()}_extract"
    out_dir = f"data/raw/{folder_name}"

    # Check if the zip file exists to avoid crashing
    if not os.path.exists(zip_path):
        print(f"Warning: '{zip_path}' not found. Skipping...")
        continue

    # Create output directory
    os.makedirs(out_dir, exist_ok=True)

    # Extract the zip
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)

    # Find all images regardless of nested folder
    images = []
    for root, _, files in os.walk(extract_to):
        for f in sorted(files):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(root, f))

    # Copy and rename images sequentially
    for i, src in enumerate(sorted(images), start=1):
        # Keep the original file extension to prevent file corruption
        _, ext = os.path.splitext(src)

        # Format the new name (e.g., book_001.jpg)
        new_name = f"{category.lower()}_{i:03d}{ext.lower()}"
        dst = os.path.join(out_dir, new_name)

        shutil.copy(src, dst)

    print(f"Copied and renamed {len(images)} images from '{zip_path}' into '{out_dir}'")

Copied and renamed 35 images from 'Book.zip' into 'data/raw/book'
Copied and renamed 38 images from 'Newspaper.zip' into 'data/raw/newspaper'
Copied and renamed 99 images from 'Signboard.zip' into 'data/raw/signboard'


In [35]:
import csv
import os

LABELS_CSV = 'data/labels.csv'
manual_folders = ['newspaper', 'books', 'signboards', 'other']

existing_rows = []
existing_paths = set()
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, 'r', encoding='utf-8') as f:
        existing_rows = list(csv.DictReader(f))
        existing_paths = {row['image'] for row in existing_rows}

new_rows = []
for folder in manual_folders:
    folder_path = f'data/raw/{folder}'
    if not os.path.exists(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = f'{folder_path}/{fname}'
        if img_path in existing_paths:
            continue  # already labeled, don't duplicate
        new_rows.append({'image': img_path, 'text': 'FILL_IN_URDU_TEXT_HERE'})

all_rows = existing_rows + new_rows
with open(LABELS_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['image', 'text'])
    writer.writeheader()
    writer.writerows(all_rows)

print(f'Added {len(new_rows)} new placeholder rows for manually uploaded images.')
print(f'labels.csv now has {len(all_rows)} total entries.')
print(f'\n{sum(1 for r in all_rows if r["text"] == "FILL_IN_URDU_TEXT_HERE")} rows '
      f'still need manual labeling -- open data/labels.csv and fill in the real text.')
print('\nTip: re-run this cell any time after adding more images -- it only')
print('adds rows for files it hasn\'t seen before, so it\'s safe to run repeatedly.')

Added 38 new placeholder rows for manually uploaded images.
labels.csv now has 168 total entries.

38 rows still need manual labeling -- open data/labels.csv and fill in the real text.

Tip: re-run this cell any time after adding more images -- it only
adds rows for files it hasn't seen before, so it's safe to run repeatedly.


In [36]:
import csv
import os

print("=" * 50)
print("FOLDER STRUCTURE & IMAGE COUNTS")
print("=" * 50)
total_images = 0
for folder in ['newspaper', 'books', 'signboards', 'synthetic', 'other']:
    folder_path = f'data/raw/{folder}'
    if os.path.exists(folder_path):
        count = len([f for f in os.listdir(folder_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total_images += count
        print(f'  {folder:12s}: {count} images')
    else:
        print(f'  {folder:12s}: folder missing!')

print(f'\nTOTAL IMAGES: {total_images}  (target: 100+)')

print("\n" + "=" * 50)
print("LABELS.CSV STATUS")
print("=" * 50)
n_placeholder = 0
if os.path.exists('data/labels.csv'):
    with open('data/labels.csv', 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    n_filled = sum(1 for r in rows if r['text'] != 'FILL_IN_URDU_TEXT_HERE')
    n_placeholder = len(rows) - n_filled
    print(f'  Total rows: {len(rows)}')
    print(f'  Labeled: {n_filled}')
    print(f'  Still need manual labeling: {n_placeholder}')
    print(f'\n  First 3 rows:')
    for r in rows[:3]:
        print(f'    {r["image"]} -> {r["text"][:40]}')
else:
    print('  labels.csv not found!')

if total_images >= 100 and n_placeholder == 0:
    print('\n✅ Ready to commit to GitHub.')
elif total_images >= 100:
    print(f'\n⚠️  Image count OK, but {n_placeholder} labels still need filling in.')
else:
    print(f'\n⚠️  Need {100 - total_images} more images.')

FOLDER STRUCTURE & IMAGE COUNTS
  newspaper   : 38 images
  books       : 0 images
  signboards  : 0 images
  synthetic   : 10 images
  other       : 60 images

TOTAL IMAGES: 108  (target: 100+)

LABELS.CSV STATUS
  Total rows: 168
  Labeled: 130
  Still need manual labeling: 38

  First 3 rows:
    data/raw/other/utrset_000.png -> مل سکتا ہے،’’
    data/raw/other/utrset_001.png -> لفظوں کی یہی عظمت اور ان کی بہی شاہانہ م
    data/raw/other/utrset_002.png -> پہچان کر عہد الزبتھ کے ڈراما نگاروں اور 

⚠️  Image count OK, but 38 labels still need filling in.
